In [ ]:
if CLEANUP_MODE:
    print("")
    print("🧹 Cleanup Mode: Removing test data to keep platform clean...")
    print("⏳ Starting cleanup in 3 seconds...")
    time.sleep(3)
    
    print("🧹 O-Nakala Test Data Cleanup Tool")
    print("=" * 50)
    
    # Load data to cleanup
    datasets = load_upload_results()
    collections = load_collection_results()
    
    if not datasets and not collections:
        print("ℹ️  No test data found to cleanup.")
        print("   Make sure you're in the directory with upload_results.csv")
    else:
        # Show what will be cleaned up
        print(f"📋 Found test data to cleanup:")
        if datasets:
            print(f"   📊 {len(datasets)} datasets")
        if collections:
            print(f"   📁 {len(collections)} collections")
        
        print(f"\n🔑 Using API key: {API_KEY[:8]}...")
        
        # Cleanup collections first (they might reference datasets)
        success_count = 0
        error_count = 0
        
        if collections:
            print(f"\n🗂️  Cleaning up {len(collections)} collections...")
            for i, collection_id in enumerate(collections, 1):
                print(f"   {i}/{len(collections)} Deleting collection {collection_id}...", end=" ")
                success, message = delete_collection(API_KEY, collection_id)
                if success:
                    print(f"✅ {message}")
                    success_count += 1
                else:
                    print(f"❌ {message}")
                    error_count += 1
                time.sleep(0.5)  # Be gentle with the API
        
        # Cleanup datasets
        if datasets:
            print(f"\n📊 Cleaning up {len(datasets)} datasets...")
            for i, dataset_id in enumerate(datasets, 1):
                print(f"   {i}/{len(datasets)} Deleting dataset {dataset_id}...", end=" ")
                success, message = delete_dataset(API_KEY, dataset_id)
                if success:
                    print(f"✅ {message}")
                    success_count += 1
                else:
                    print(f"❌ {message}")
                    error_count += 1
                time.sleep(0.5)  # Be gentle with the API
        
        # Summary
        print(f"\n📊 Cleanup Summary:")
        print(f"   ✅ Successfully cleaned: {success_count}")
        print(f"   ❌ Errors: {error_count}")
        
        if error_count == 0:
            print(f"\n🎉 All test data cleaned up successfully!")
            print(f"   Test platform is now clean for the next user.")
        else:
            print(f"\n⚠️  Some items couldn't be deleted (might already be gone).")
        
        # Backup result files
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_files = ['upload_results.csv', 'collections_output.csv', 'quality_report.json', 'auto_data_modifications.csv']
        
        print(f"\n📁 Backing up result files...")
        for file in backup_files:
            if os.path.exists(file):
                backup_name = f"{file}.backup_{timestamp}"
                os.rename(file, backup_name)
                print(f"   📄 {file} → {backup_name}")
        
        print(f"✅ Result files backed up with timestamp {timestamp}")
        
        print("")
        print("✨ Test platform cleaned! Ready for next user.")
else:
    print("")
    print("ℹ️  Cleanup mode disabled. Test data remains on platform.")
    print("   To cleanup manually, set CLEANUP_MODE = True and re-run this cell.")

print("")
print("🏆 ULTIMATE WORKFLOW: ALL 7 STEPS COMPLETED SUCCESSFULLY!")

In [ ]:
# Cleanup functionality (incorporated from cleanup_test_data.py)
def load_upload_results(file_path=None):
    """Load dataset IDs from upload results."""
    dataset_ids = []
    
    # Try multiple possible filenames
    possible_files = file_path and [file_path] or [
        "upload_results.csv", 
        "fresh_upload_results.csv", 
        "workshop_upload_results.csv"
    ]
    
    upload_file = None
    for filename in possible_files:
        if os.path.exists(filename):
            upload_file = filename
            break
    
    if not upload_file:
        print(f"❌ No upload results file found!")
        print(f"   Looked for: {', '.join(possible_files)}")
        return dataset_ids
    
    print(f"📄 Found upload results: {upload_file}")
    with open(upload_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('identifier') and row.get('status') == 'OK':
                dataset_ids.append(row['identifier'])
    
    return dataset_ids

def load_collection_results(file_path="collections_output.csv"):
    """Load collection IDs from collection results."""
    collection_ids = []
    if not os.path.exists(file_path):
        print(f"ℹ️  Collection results file '{file_path}' not found (this is normal if collections weren't created)")
        return collection_ids
    
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('collection_id') and row.get('creation_status') == 'SUCCESS':
                collection_ids.append(row['collection_id'])
    
    return collection_ids

def delete_dataset(api_key, dataset_id, api_url="https://apitest.nakala.fr"):
    """Delete a single dataset."""
    headers = {'X-API-KEY': api_key}
    url = f"{api_url}/datas/{dataset_id}"
    
    try:
        response = requests.delete(url, headers=headers)
        if response.status_code == 204:
            return True, "Deleted successfully"
        elif response.status_code == 404:
            return True, "Already deleted or not found"
        else:
            return False, f"HTTP {response.status_code}: {response.text}"
    except Exception as e:
        return False, f"Error: {str(e)}"

def delete_collection(api_key, collection_id, api_url="https://apitest.nakala.fr"):
    """Delete a single collection."""
    headers = {'X-API-KEY': api_key}
    url = f"{api_url}/collections/{collection_id}"
    
    try:
        response = requests.delete(url, headers=headers)
        if response.status_code == 204:
            return True, "Deleted successfully"
        elif response.status_code == 404:
            return True, "Already deleted or not found"
        else:
            return False, f"HTTP {response.status_code}: {response.text}"
    except Exception as e:
        return False, f"Error: {str(e)}"

print("🛠️  Cleanup functions defined")

## Optional: Cleanup Test Data

This section incorporates the cleanup functionality to remove test data from the platform.

In [ ]:
print("\n🎯 Step 7/7: Results Summary")
print("----------------------------")

# Count results
datasets_count = 0
collections_count = 0
first_dataset = "N/A"

try:
    with open('upload_results.csv', 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        datasets_count = len(rows)
        if rows:
            first_dataset = rows[0].get('identifier', 'N/A')
except FileNotFoundError:
    datasets_count = 0

try:
    with open('collections_output.csv', 'r') as f:
        collections_count = sum(1 for line in f) - 1  # Subtract header
except FileNotFoundError:
    collections_count = 0

print(f"📊 Datasets Created: {datasets_count}")
print(f"📁 Collections Created: {collections_count}")
print(f"📈 Quality Report: quality_report.json")
print(f"🔗 First Dataset: {first_dataset}")

print("")
print("🎉 COMPLETE WORKFLOW FINISHED SUCCESSFULLY!")
print("===========================================")
print("📄 All 7 steps completed:")
print("   ✅ 1. Environment Setup")
print(f"   ✅ 2. Data Upload ({datasets_count} datasets)")
print(f"   ✅ 3. Collection Creation ({collections_count} collections)")
print("   ✅ 4. Auto-Enhancement (intelligent metadata)")
print("   ✅ 5. Dataset Curation (100% success)")
print("   ✅ 6. Collection Curation (100% success)")
print("   ✅ 7. Quality Analysis (comprehensive report)")

## Step 7/7: Results Summary

In [ ]:
print("\n📊 Step 6/7: Quality Analysis")
print("-----------------------------")

# Run the quality analysis command
quality_cmd = [
    'o-nakala-curator',
    '--api-key', API_KEY,
    '--quality-report',
    '--scope', 'datasets',
    '--output', 'quality_report.json'
]

try:
    result = subprocess.run(quality_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
    if result.stderr:
        print("Warnings/Errors:", result.stderr)
    
    print("✅ Comprehensive quality report generated")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Quality analysis failed: {e}")
    print(f"Error output: {e.stderr}")

## Step 6/7: Quality Analysis

In [ ]:
print("\n📁 Step 5/7: Collection Metadata Curation")
print("-----------------------------------------")

# Run the collection curation command if collection modifications exist
if os.path.exists('auto_collection_modifications.csv'):
    collection_curation_cmd = [
        'o-nakala-curator',
        '--api-key', API_KEY,
        '--batch-modify', 'auto_collection_modifications.csv',
        '--scope', 'collections'
    ]
    
    try:
        result = subprocess.run(collection_curation_cmd, capture_output=True, text=True, check=True)
        print(result.stdout)
        if result.stderr:
            print("Warnings/Errors:", result.stderr)
        
        print("✅ Collection metadata professionally enhanced")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Collection curation failed: {e}")
        print(f"Error output: {e.stderr}")
else:
    print("ℹ️  No collection modifications to apply")
    print("✅ Collection curation step completed")

## Step 5/7: Collection Metadata Curation

In [ ]:
print("\n✨ Step 4/7: Dataset Metadata Curation")
print("--------------------------------------")

# Run the dataset curation command
dataset_curation_cmd = [
    'o-nakala-curator',
    '--api-key', API_KEY,
    '--batch-modify', 'auto_data_modifications.csv',
    '--scope', 'datasets'
]

try:
    result = subprocess.run(dataset_curation_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
    if result.stderr:
        print("Warnings/Errors:", result.stderr)
    
    print("✅ Dataset metadata professionally enhanced")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Dataset curation failed: {e}")
    print(f"Error output: {e.stderr}")

## Step 4/7: Dataset Metadata Curation

In [ ]:
# Incorporate create_collection_modifications.py functionality
def create_collection_modifications(collections_file="collections_output.csv", output_file="auto_collection_modifications.csv"):
    """Create collection modification CSV from collections output."""
    
    if not os.path.exists(collections_file):
        print(f"❌ Collections output file '{collections_file}' not found!")
        return None
    
    modifications = []
    
    # Read collections output
    with open(collections_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['collection_id'] and row['creation_status'] == 'SUCCESS':
                collection_id = row['collection_id']
                title = row['collection_title']
                
                # Create enhanced modifications based on collection content
                if 'Code' in title and 'Data' in title:
                    new_title = "fr:Collection Code et Données Avancée|en:Advanced Code and Data Collection"
                    new_desc = "fr:Collection professionnelle regroupant scripts d'analyse et jeux de données de recherche validés avec documentation complète|en:Professional collection grouping analysis scripts and validated research datasets with complete documentation"
                    new_keywords = "fr:code;données;scripts;analyse;recherche;python;r;validation|en:code;data;scripts;analysis;research;python;r;validation"
                    
                elif 'Documents' in title:
                    new_title = "fr:Collection Documentation Académique|en:Academic Documentation Collection"
                    new_desc = "fr:Documentation de recherche exhaustive incluant méthodologie, protocoles et analyses approfondies pour publication scientifique|en:Comprehensive research documentation including methodology, protocols and in-depth analyses for scientific publication"
                    new_keywords = "fr:documentation;recherche;méthodologie;protocoles;publication;académique|en:documentation;research;methodology;protocols;publication;academic"
                    
                elif 'Multimedia' in title or 'Multimédia' in title:
                    new_title = "fr:Collection Multimédia Professionnelle|en:Professional Multimedia Collection"
                    new_desc = "fr:Ressources visuelles et de présentation de haute qualité pour communication scientifique et diffusion de résultats de recherche|en:High-quality visual and presentation resources for scientific communication and research dissemination"
                    new_keywords = "fr:multimédia;images;présentations;communication;diffusion;recherche|en:multimedia;images;presentations;communication;dissemination;research"
                    
                else:
                    # Generic enhancement for other collections
                    new_title = f"fr:Collection Recherche Professionnelle|en:Professional Research Collection"
                    new_desc = "fr:Collection organisée de ressources de recherche avec métadonnées enrichies pour faciliter la découverte et l'accès|en:Organized collection of research resources with enriched metadata to facilitate discovery and access"
                    new_keywords = "fr:recherche;collection;métadonnées;accès;découverte|en:research;collection;metadata;access;discovery"
                
                modifications.append({
                    'id': collection_id,
                    'action': 'modify',
                    'new_title': new_title,
                    'new_description': new_desc,
                    'new_keywords': new_keywords
                })
    
    if not modifications:
        print("❌ No collections found to modify!")
        return None
    
    # Write modifications CSV
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['id', 'action', 'new_title', 'new_description', 'new_keywords']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(modifications)
    
    print(f"✅ Created {len(modifications)} collection modifications in '{output_file}'")
    return output_file

# Generate collection modifications if collections exist
if os.path.exists('collections_output.csv'):
    create_collection_modifications('collections_output.csv')
else:
    print("ℹ️  No collections to modify (collections_output.csv not found)")

print("✅ Professional metadata enhancements generated")

In [ ]:
print("\n🤖 Step 3/7: Auto-Enhancement Generation")
print("----------------------------------------")

# Incorporate create_modifications.py functionality
def create_modifications_from_upload(upload_file="upload_results.csv", output_file="auto_data_modifications.csv"):
    """Create modification CSV automatically from upload results."""
    
    if not os.path.exists(upload_file):
        print(f"❌ Upload results file '{upload_file}' not found!")
        return None
    
    modifications = []
    
    # Read upload results
    with open(upload_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['identifier'] and row['status'] == 'OK':
                # Extract title for context
                title = row['title']
                
                # Create enhanced modifications based on title content
                if 'Images' in title or 'images' in title:
                    new_title = "fr:Images de Site Web Optimisées|en:Optimized Website Images"
                    new_desc = "fr:Collection d'images photographiques professionnelles pour documentation de site|en:Collection of professional photographic images for site documentation"
                    new_keywords = "fr:images;photographie;site;documentation|en:images;photography;site;documentation"
                    
                elif 'Scripts' in title or 'Code' in title or 'code' in title:
                    new_title = "fr:Scripts d'Analyse Professionnels|en:Professional Analysis Scripts"
                    new_desc = "fr:Scripts Python et R optimisés pour l'analyse de données de recherche avec documentation complète|en:Optimized Python and R scripts for research data analysis with complete documentation"
                    new_keywords = "fr:code;scripts;analyse;recherche;python;r|en:code;scripts;analysis;research;python;r"
                    
                elif 'Présentations' in title or 'Presentations' in title:
                    new_title = "fr:Matériaux de Présentation Académiques|en:Academic Presentation Materials"
                    new_desc = "fr:Supports de communication pour conférences et réunions académiques professionnels|en:Professional communication materials for academic conferences and meetings"
                    new_keywords = "fr:présentations;académique;conférences;communication|en:presentations;academic;conferences;communication"
                    
                elif 'Documents' in title or 'documents' in title:
                    new_title = "fr:Documentation de Recherche Complète|en:Complete Research Documentation"
                    new_desc = "fr:Documentation académique exhaustive incluant protocoles, méthodologie et analyses|en:Comprehensive academic documentation including protocols, methodology and analyses"
                    new_keywords = "fr:documentation;recherche;protocoles;méthodologie|en:documentation;research;protocols;methodology"
                    
                elif 'Données' in title or 'Data' in title:
                    new_title = "fr:Données de Recherche Validées|en:Validated Research Data"
                    new_desc = "fr:Jeux de données collectés, traités et validés pour analyse statistique approfondie|en:Data sets collected, processed and validated for in-depth statistical analysis"
                    new_keywords = "fr:données;recherche;analyse;statistiques;validé|en:data;research;analysis;statistics;validated"
                    
                else:
                    # Generic enhancement
                    new_title = f"fr:Ressource Enrichie|en:Enhanced Resource"
                    new_desc = "fr:Ressource académique améliorée avec métadonnées professionnelles|en:Enhanced academic resource with professional metadata"
                    new_keywords = "fr:ressource;académique;enrichi|en:resource;academic;enhanced"
                
                modifications.append({
                    'id': row['identifier'],
                    'action': 'modify',
                    'new_title': new_title,
                    'new_description': new_desc,
                    'new_keywords': new_keywords
                })
    
    if not modifications:
        print("❌ No successful uploads found to create modifications!")
        return None
    
    # Write modifications CSV
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['id', 'action', 'new_title', 'new_description', 'new_keywords']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(modifications)
    
    print(f"✅ Created {output_file} with {len(modifications)} modifications")
    return output_file

# Generate dataset modifications
create_modifications_from_upload('upload_results.csv')

## Step 3/7: Auto-Enhancement Generation

This step incorporates the Python scripts to generate professional metadata enhancements.

In [ ]:
print("\n📁 Step 2/7: Collection Creation")
print("--------------------------------")

# Run the collection creation command
collection_cmd = [
    'o-nakala-collection',
    '--api-key', API_KEY,
    '--from-upload-output', 'upload_results.csv',
    '--from-folder-collections', 'folder_collections.csv',
    '--collection-report', 'collections_output.csv'
]

try:
    result = subprocess.run(collection_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
    if result.stderr:
        print("Warnings/Errors:", result.stderr)
    
    # Count created collections
    collections_created = 0
    try:
        with open('collections_output.csv', 'r') as f:
            collections_created = sum(1 for line in f) - 1  # Subtract header
    except FileNotFoundError:
        collections_created = 0
        
    print(f"✅ Collections created - {collections_created} collections")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Collection creation failed: {e}")
    print(f"Error output: {e.stderr}")
    collections_created = 0

## Step 2/7: Collection Creation

In [ ]:
print("\n📤 Step 1/7: Data Upload")
print("------------------------")

# Run the upload command
upload_cmd = [
    'o-nakala-upload',
    '--api-key', API_KEY,
    '--dataset', 'folder_data_items.csv',
    '--mode', 'folder',
    '--folder-config', 'folder_data_items.csv',
    '--base-path', '.',
    '--output', 'upload_results.csv'
]

try:
    result = subprocess.run(upload_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
    if result.stderr:
        print("Warnings/Errors:", result.stderr)
    
    # Count uploaded datasets
    with open('upload_results.csv', 'r') as f:
        dataset_count = sum(1 for line in f) - 1  # Subtract header
        
    print(f"✅ Upload completed - {dataset_count} datasets created")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Upload failed: {e}")
    print(f"Error output: {e.stderr}")

## Step 1/7: Data Upload

In [ ]:
# Configuration
import os
import subprocess
import csv
import json
import requests
import time
from datetime import datetime
from IPython.display import display, HTML

# Set your API key here or use environment variable
API_KEY = os.environ.get('NAKALA_API_KEY', '33170cfe-f53c-550b-5fb6-4814ce981293')  # Test key
CLEANUP_MODE = True  # Set to True to enable cleanup after workflow

print(f"🔑 Using API Key: {API_KEY[:10]}...")
if CLEANUP_MODE:
    print("🧹 Cleanup mode enabled - test data will be removed after demonstration")

# Set environment variable for CLI commands
os.environ['NAKALA_API_KEY'] = API_KEY

## Configuration

# Ultimate O-Nakala Core Workflow

Complete Steps 1-7 with optional cleanup functionality.

This notebook combines all the workflow steps from the shell script version:
1. Data Upload
2. Collection Creation  
3. Auto-Enhancement Generation
4. Dataset Metadata Curation
5. Collection Metadata Curation
6. Quality Analysis
7. Results Summary
8. Optional Cleanup